# 03 — Huấn luyện DNN, LSTM, CNN

Ba client **UNSW + BoT + CIC**; 39 feature. CIC giữ đúng 1.000.000 mẫu thật (500.000 benign + 500.000 attack). Mọi mô hình dùng **CSV raw → Log → Z-score riêng từng client → clip [-8,8]**. Mean/std chỉ fit trên tập fit; đóng băng khi dùng validation/test. Split và nhãn được giữ nguyên. Kết quả lưu ở `runs/main/`.

Ở mỗi round FL, server gộp cập nhật của UNSW/BoT/CIC với trọng số bằng nhau 1/3 rồi dùng Adam cập nhật model toàn cục. Chính sách này giống notebook tuning 02.

In [1]:
from pathlib import Path
import os, sys, json
if os.name == 'nt':
    raise RuntimeError('Mở start_notebooks.cmd để dùng kernel WSL /opt/flids-tf.')
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p/'notebook_workflow.py').is_file() and (p/'configs/tuning.json').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Mở notebook trong FL_IDS.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
CONFIG = PROJECT/'configs/tuning.json'
from notebook_workflow import settings, budget
C = settings(CONFIG)
RUN_ROOT = PROJECT/C['run_directory']
print('Project:', PROJECT)
print('Python:', sys.executable)
print('Run:', RUN_ROOT)


Project: /mnt/c/Users/admin/Downloads/FL_IDS
Python: /opt/flids-tf/bin/python
Run: /mnt/c/Users/admin/Downloads/FL_IDS/runs/main


## Huấn luyện bằng tham số được chọn từ validation

Ba Local + Centralized + FL. Ngân sách chung: FL 15 round × 3 local epoch; Local/Centralized tối đa 45 epoch, early stopping val_loss patience 5. Áp dụng cho cả tuning và training cuối; cấu hình trong configs/tuning.json.

In [2]:
from notebook_workflow import run_experiments, selected_params
SCENARIOS = None  # hoặc ['federated'], ['centralized'], ['local_unsw','local_bot','local_cic']

In [3]:
display(selected_params(C,'dnn'))
results_dnn = run_experiments('dnn', CONFIG, scenarios=SCENARIOS)
display(results_dnn)

{'study_identity': 'a918907e0ec7b70ed015ce689de6904b02ed6f517bb383caba216fd40749af66',
 'model': 'dnn',
 'trial': 9,
 'params': {'architecture': '16-8-4',
  'dropout': 0.4,
  'batch_size': 4096,
  'client_learning_rate': 0.002,
  'server_learning_rate': 0.01},
 'rank': [0.7695237209251039, 0.8481094127111827, 0.7709171313391866],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'objective': 'mean of 12 three-client comparison F1 values; Local/Centralized at minimum val_loss epoch, FL at final round'}

2026-10-24 08:45:25.346170: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 08:45:25.346237: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 08:45:25.346265: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 08:45:27.516413: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 08:45:27.516469: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register f

,model,scenario,source,test_dataset,samples,accuracy,precision,recall,f1,far,auc,tn,fp,fn,tp
0,dnn,centralized,all_clients,unsw,57032,0.991741,0.989869,0.993653,0.991757,0.010170,0.995116,28226,290,181,28335
1,dnn,centralized,all_clients,bot,81023,0.965553,0.997914,0.933055,0.964396,0.001950,0.994791,40433,79,2712,37799
2,dnn,centralized,all_clients,cic,300000,0.976873,0.997213,0.956420,0.976390,0.002673,0.985847,149599,401,6537,143463
3,dnn,federated,all_clients,unsw,57032,0.842299,0.925427,0.744600,0.825223,0.060001,0.906755,26805,1711,7283,21233
4,dnn,federated,all_clients,bot,81023,0.954605,0.978289,0.929846,0.953452,0.020636,0.979922,39676,836,2842,37669
5,dnn,federated,all_clients,cic,300000,0.900773,0.892392,0.911453,0.901822,0.109907,0.943370,133514,16486,13282,136718
6,dnn,local,bot,unsw,57032,0.844035,0.865096,0.815191,0.839403,0.127122,0.891705,24891,3625,5270,23246
7,dnn,local,bot,bot,81023,0.968996,0.995359,0.942386,0.968149,0.004394,0.996686,40334,178,2334,38177
8,dnn,local,bot,cic,300000,0.726177,0.726205,0.726113,0.726159,0.273760,0.656884,108936,41064,41083,108917
9,dnn,local,cic,unsw,57032,0.476890,0.471373,0.380523,0.421104,0.426743,0.445846,16347,12169,17665,10851


In [4]:
display(selected_params(C,'lstm'))
results_lstm = run_experiments('lstm', CONFIG, scenarios=SCENARIOS)
display(results_lstm)

{'study_identity': 'a918907e0ec7b70ed015ce689de6904b02ed6f517bb383caba216fd40749af66',
 'model': 'lstm',
 'trial': 7,
 'params': {'architecture': '16-8-4',
  'dropout': 0.3,
  'batch_size': 1024,
  'client_learning_rate': 0.001,
  'server_learning_rate': 0.075},
 'rank': [0.7585189050395194, 0.7279253409906676, 0.7606622087878455],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'objective': 'mean of 12 three-client comparison F1 values; Local/Centralized at minimum val_loss epoch, FL at final round'}

2026-10-24 08:51:37.215724: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 08:51:37.215775: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 08:51:37.215809: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 08:51:39.292775: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 08:51:39.292844: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register f

,model,scenario,source,test_dataset,samples,accuracy,precision,recall,f1,far,auc,tn,fp,fn,tp
0,dnn,centralized,all_clients,unsw,57032,0.991741,0.989869,0.993653,0.991757,0.010170,0.995116,28226,290,181,28335
1,dnn,centralized,all_clients,bot,81023,0.965553,0.997914,0.933055,0.964396,0.001950,0.994791,40433,79,2712,37799
2,dnn,centralized,all_clients,cic,300000,0.976873,0.997213,0.956420,0.976390,0.002673,0.985847,149599,401,6537,143463
3,dnn,federated,all_clients,unsw,57032,0.842299,0.925427,0.744600,0.825223,0.060001,0.906755,26805,1711,7283,21233
4,dnn,federated,all_clients,bot,81023,0.954605,0.978289,0.929846,0.953452,0.020636,0.979922,39676,836,2842,37669
5,dnn,federated,all_clients,cic,300000,0.900773,0.892392,0.911453,0.901822,0.109907,0.943370,133514,16486,13282,136718
6,dnn,local,bot,unsw,57032,0.844035,0.865096,0.815191,0.839403,0.127122,0.891705,24891,3625,5270,23246
7,dnn,local,bot,bot,81023,0.968996,0.995359,0.942386,0.968149,0.004394,0.996686,40334,178,2334,38177
8,dnn,local,bot,cic,300000,0.726177,0.726205,0.726113,0.726159,0.273760,0.656884,108936,41064,41083,108917
9,dnn,local,cic,unsw,57032,0.476890,0.471373,0.380523,0.421104,0.426743,0.445846,16347,12169,17665,10851


In [5]:
display(selected_params(C,'cnn'))
results_cnn = run_experiments('cnn', CONFIG, scenarios=SCENARIOS)
display(results_cnn)

{'study_identity': 'a918907e0ec7b70ed015ce689de6904b02ed6f517bb383caba216fd40749af66',
 'model': 'cnn',
 'trial': 10,
 'params': {'architecture': '4-8-4',
  'dropout': 0.1,
  'batch_size': 1024,
  'client_learning_rate': 0.0003,
  'server_learning_rate': 0.01,
  'kernel_size': 3},
 'rank': [0.8224010022235628, 0.8744021427204898, 0.8444676932676125],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'objective': 'mean of 12 three-client comparison F1 values; Local/Centralized at minimum val_loss epoch, FL at final round'}

2026-10-24 09:09:39.403703: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 09:09:39.403752: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 09:09:39.403771: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 09:09:41.566814: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 09:09:41.566871: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register f

,model,scenario,source,test_dataset,samples,accuracy,precision,recall,f1,far,auc,tn,fp,fn,tp
0,cnn,centralized,all_clients,unsw,57032,0.993249,0.990891,0.995652,0.993266,0.009153,0.997178,28255,261,124,28392
1,cnn,centralized,all_clients,bot,81023,0.971860,0.995850,0.947669,0.971162,0.003949,0.993564,40352,160,2120,38391
2,cnn,centralized,all_clients,cic,300000,0.976007,0.995084,0.956740,0.975535,0.004727,0.986387,149291,709,6489,143511
3,cnn,federated,all_clients,unsw,57032,0.907052,0.937112,0.872668,0.903742,0.058564,0.966969,26846,1670,3631,24885
4,cnn,federated,all_clients,bot,81023,0.924762,0.917303,0.933697,0.925428,0.084173,0.967133,37102,3410,2686,37825
5,cnn,federated,all_clients,cic,300000,0.879730,0.873738,0.887747,0.880686,0.128287,0.955374,130757,19243,16838,133162
6,cnn,local,bot,unsw,57032,0.819487,0.882200,0.737446,0.803354,0.098471,0.872799,25708,2808,7487,21029
7,cnn,local,bot,bot,81023,0.967540,0.994053,0.940707,0.966645,0.005628,0.997784,40284,228,2402,38109
8,cnn,local,bot,cic,300000,0.780823,0.858580,0.672400,0.754170,0.110753,0.781304,133387,16613,49140,100860
9,cnn,local,cic,unsw,57032,0.478661,0.475938,0.422079,0.447393,0.464757,0.464346,15263,13253,16480,12036
